In [2]:
from urllib.request import urlopen
from PIL import Image
from pathlib import Path
from torch import optim
from tqdm import tqdm

import timm
import os, shutil, random


import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.datasets as datasets

C:\Users\xuant\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:

data_transforms = {
    'train': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], 
                             [0.229, 0.224, 0.225])

    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], 
                             [0.229, 0.224, 0.225])
    ]),
}

In [4]:
trainset = datasets.ImageFolder(r'C:\Users\xuant\Documents\GitHub\WoodTN\data_split\train',transform=data_transforms['train'])
trainsloader = torch.utils.data.DataLoader(trainset, batch_size=32, shuffle=True, num_workers=4, pin_memory= True)

testset = datasets.ImageFolder(r'C:\Users\xuant\Documents\GitHub\WoodTN\data_split\val',transform=data_transforms['val'])
testloader = torch.utils.data.DataLoader(testset, batch_size=32, shuffle=False, num_workers=4, pin_memory= True)

In [5]:
datasets_sizes = {
    'train': len(trainset),
    'val': len(testset)
}
data_loaders = {
    'train': trainsloader,
    'val': testloader
}

In [6]:
datasets_sizes

{'train': 23792, 'val': 4751}

In [7]:
trainset.classes

['Bach dan Lang Son',
 'Bach dan Thai Nguyen',
 'Gao',
 'Gioi',
 'Keo lai',
 'Keo tai tuong',
 'Lat hoa',
 'Lim xanh',
 'Mo Thai Nguyen',
 'Mo Tuyen Quang',
 'Que',
 'Tech',
 'Thong',
 'Xoan ta Dai Tu',
 'Xoan ta Dinh Hoa',
 'other']

In [8]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
num_classes = len(trainset.classes)

model = timm.create_model('mvitv2_small', pretrained=True)
print('model.head:', model.head)
# Replace the classifier safely. For MViTv2 the head is Sequential with a named 'fc' Linear
if hasattr(model.head, 'fc') and isinstance(model.head.fc, nn.Linear):
    model.head.fc = nn.Linear(model.head.fc.in_features, num_classes)
elif isinstance(model.head, nn.Linear):
    model.head = nn.Linear(model.head.in_features, num_classes)
elif isinstance(model.head, nn.Sequential):
    # find last Linear in Sequential and replace it
    for i in range(len(model.head) - 1, -1, -1):
        if isinstance(model.head[i], nn.Linear):
            in_f = model.head[i].in_features
            model.head[i] = nn.Linear(in_f, num_classes)
            break
else:
    # fallback to timm helper if available
    if hasattr(model, 'reset_classifier'):
        model.reset_classifier(num_classes)

print('Updated model.head:', model.head)
model = model.to(device)

model.head: Sequential(
  (drop): Dropout(p=0.0, inplace=False)
  (fc): Linear(in_features=768, out_features=1000, bias=True)
)
Updated model.head: Sequential(
  (drop): Dropout(p=0.0, inplace=False)
  (fc): Linear(in_features=768, out_features=16, bias=True)
)


In [9]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr = 1e-4, weight_decay = 1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max = 10)

In [ ]:
# Training loop with mixed precision, checkpointing and early stopping
import time

best_val_acc = 0.0
use_amp = (device.type == 'cuda')
scaler = torch.cuda.amp.GradScaler() if use_amp else None
save_dir = 'checkpoints'
os.makedirs(save_dir, exist_ok=True)

num_epochs = 20
# Early stopping params
patience = 5          
min_delta = 1e-4
epochs_no_improve = 0

for epoch in range(num_epochs):
    t0 = time.time()
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for imgs, labels in tqdm(trainsloader, desc=f'Train Epoch {epoch+1}/{num_epochs}'):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()

        if use_amp:
            with torch.cuda.amp.autocast():
                out = model(imgs)
                loss = criterion(out, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            out = model(imgs)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()

        batch_size = imgs.size(0)
        total_loss += loss.item() * batch_size
        _, preds = out.max(dim=1)
        correct += (preds == labels).sum().item()
        total += batch_size

    train_loss = total_loss / total if total > 0 else 0.0
    train_acc = correct / total if total > 0 else 0.0

    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in tqdm(testloader, desc='Val', leave=False):
            imgs, labels = imgs.to(device), labels.to(device)
            if use_amp:
                with torch.cuda.amp.autocast():
                    out = model(imgs)
                    loss = criterion(out, labels)
            else:
                out = model(imgs)
                loss = criterion(out, labels)

            bsize = imgs.size(0)
            val_loss += loss.item() * bsize
            _, preds = out.max(dim=1)
            val_correct += (preds == labels).sum().item()
            val_total += bsize

    val_loss_avg = val_loss / val_total if val_total > 0 else 0.0
    val_acc = val_correct / val_total if val_total > 0 else 0.0

    epoch_time = time.time() - t0
    print(f"Epoch {epoch+1}/{num_epochs}  Time: {epoch_time:.1f}s  Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.4f}  Val Loss: {val_loss_avg:.4f}  Val Acc: {val_acc:.4f}")

    # Save best model by validation accuracy
    if val_acc > best_val_acc + min_delta:
        best_val_acc = val_acc
        epochs_no_improve = 0
        ckpt_path = os.path.join(save_dir, f'mvitv2_best_acc_{best_val_acc:.4f}.pth')
        torch.save(model.state_dict(), ckpt_path)
        print(f"Saved best model to {ckpt_path}")
    else:
        epochs_no_improve += 1
        print(f"No improvement for {epochs_no_improve} epoch(s)")

    # Step LR scheduler (per epoch)
    scheduler.step()

    # Early stopping check
    if epochs_no_improve >= patience:
        print(f"Early stopping triggered. No improvement for {patience} epochs.\nBest val acc: {best_val_acc:.4f}")
        break

# Optionally save final model
torch.save(model.state_dict(), os.path.join(save_dir, 'mvitv2_last.pth'))